# Urhobo TTS Engine — Phase 9: Model Fine-Tuning (Kaggle GPU)

This notebook fine-tunes **Meta MMS-TTS Yoruba** (`facebook/mms-tts-yor`) on **2.52 hours of studio-grade Urhobo speech** with our custom **70-token Urhobo vocabulary** on a free **Kaggle GPU (T4 x 1 or P100)**.

### Pipeline Overview:
1. **GPU Check**: Verify CUDA accelerator.
2. **Dependencies**: Install pinned `transformers`, `datasets[audio]`, `accelerate`, and `Cython`.
3. **Repo Sync**: Clone `ruxy1212/urhobo-tts` and `ylacombe/finetune-hf-vits`.
4. **Monotonic Alignment Compilation**: Build the Cython acceleration module.
5. **Urhobo Tokenizer & VITS Embedding Resizing**: Expand base vocabulary from 43 to 70 tokens (adding 'v', 'c', 'z', tone diacritics).
6. **Dataset Ingestion**: Load tone-supervised manifests (`train.jsonl`, `dev.jsonl`) at 16kHz.
7. **Fine-Tuning Execution**: Train VITS generator and discriminator with periodic checkpointing.
8. **Inference & Audio Spot-Check**: Synthesize Urhobo phrases directly in the notebook.
9. **Export & Persistence**: Package trained checkpoint into Kaggle output dataset.

### Step 1: Verify GPU Environment
Ensure accelerator is set to **GPU T4 x 1** or **GPU P100** in notebook settings (right panel), and **Internet is ON**.

In [1]:
!nvidia-smi

Thu Sep 17 03:04:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Install Pinned Dependencies

In [2]:
# Install Cython for monotonic alignment acceleration
!pip install -q Cython


### Step 3: Clone Project Repository & Training Library

In [3]:
import os
import sys

# Always start from a safe, persistent root directory
%cd /kaggle/working

# 1. Clone or pull main Urhobo TTS project (verify .git exists)
if not os.path.exists('/kaggle/working/urhobo-tts/.git'):
    !rm -rf /kaggle/working/urhobo-tts
    !git clone https://github.com/ruxy1212/urhobo-tts.git /kaggle/working/urhobo-tts
else:
    %cd /kaggle/working/urhobo-tts
    !git pull origin main
    %cd /kaggle/working

# 2. Clone or update VITS toolkit & compile monotonic alignment
if not os.path.exists('/kaggle/working/finetune-hf-vits/.git'):
    !rm -rf /kaggle/working/finetune-hf-vits
    !git clone https://github.com/ylacombe/finetune-hf-vits.git /kaggle/working/finetune-hf-vits

%cd /kaggle/working/finetune-hf-vits/monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace

# Return to Urhobo TTS working directory
%cd /kaggle/working/urhobo-tts

# 3. Discover attached Kaggle inputs and link audio files
kaggle_input = '/kaggle/input'
print('Attached Kaggle datasets:', os.listdir(kaggle_input) if os.path.exists(kaggle_input) else 'None')

target_gen = '/kaggle/working/urhobo-tts/data/processed/segments/GEN'
os.makedirs(target_gen, exist_ok=True)

if os.path.exists(kaggle_input):
    for folder in os.listdir(kaggle_input):
        candidate = os.path.join(kaggle_input, folder)
        for root, dirs, files in os.walk(candidate):
            # If uploaded as zip
            for f in files:
                if f.endswith('.zip') and any(k in f.lower() for k in ['audio', 'segment', 'urhobo']):
                    zip_path = os.path.join(root, f)
                    print(f'Extracting {zip_path}...')
                    !unzip -q -o {zip_path} -d /kaggle/working/urhobo-tts/
            # If uncompressed WAV files exist
            wavs = [f for f in files if f.endswith('.wav')]
            if wavs:
                print(f'Found {len(wavs)} WAVs in {root}, linking to {target_gen}...')
                !ln -sf {root}/*.wav {target_gen}/
                break

wav_on_disk = len([f for f in os.listdir(target_gen) if f.endswith('.wav')]) if os.path.exists(target_gen) else 0
print(f'\n=> Total Segments Ready on Disk: {wav_on_disk} WAV clips')
print(f'Models directory exists: {os.path.exists("/kaggle/working/urhobo-tts/models/urhobo_tokenizer")}')


/kaggle/working
Cloning into '/kaggle/working/urhobo-tts'...
remote: Enumerating objects: 708, done.
remote: Counting objects: 100% (708/708), done.
remote: Compressing objects: 100% (466/466), done.
remote: Total 708 (delta 403), reused 536 (delta 231), pack-reused 0 (from 0)
Receiving objects: 100% (708/708), 3.05 MiB | 9.48 MiB/s, done.
Resolving deltas: 100% (403/403), done.
Cloning into '/kaggle/working/finetune-hf-vits'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 173 (delta 81), reused 74 (delta 74), pack-reused 78 (from 1)
Receiving objects: 100% (173/173), 1.24 MiB | 3.30 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/kaggle/working/finetune-hf-vits/monotonic_align
Compiling core.pyx because it changed.
[1/1] Cythonizing core.pyx
/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str'

### Step 4: Verify Urhobo Tokenizer & Resized Embedding Matrix

In [4]:
import json
import torch
from transformers import AutoTokenizer, VitsModel

TOKENIZER_PATH = "/kaggle/working/urhobo-tts/models/urhobo_tokenizer"
BASE_MODEL = "facebook/mms-tts-yor"

# Load custom 70-token Urhobo tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
print(f"Loaded Urhobo Tokenizer: {len(tokenizer)} tokens")

# Test sample Urhobo sentence
test_text = "Avwanre vwo ẹgba vwọ kẹ Osolobrugwẹ. Mẹ́vwẹ yen rha cha."
encoded = tokenizer(test_text)
print("Sample Token IDs:", encoded["input_ids"][:15])

# Load Base VITS Model
print(f"Loading base model {BASE_MODEL}...")
model = VitsModel.from_pretrained(BASE_MODEL)
old_vocab_size = model.text_encoder.embed_tokens.weight.shape[0]
print(f"Original embedding shape: {model.text_encoder.embed_tokens.weight.shape}")

# Resize embedding table using scripts/14 logic
sys.path.append("/kaggle/working/urhobo-tts/scripts")
from importlib import import_module
prep_tok = import_module("14_prepare_urhobo_tokenizer")
model = prep_tok.resize_vits_embeddings(model, new_vocab_size=len(tokenizer))
print(f"Resized embedding shape:  {model.text_encoder.embed_tokens.weight.shape}")
print(f"Vocab expansion verified: {old_vocab_size} -> {len(tokenizer)} tokens!")

Loaded Urhobo Tokenizer: 71 tokens
Sample Token IDs: [0, 8, 0, 43, 0, 12, 0, 8, 0, 1, 0, 9, 0, 27, 0]
Loading base model facebook/mms-tts-yor...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Original embedding shape: torch.Size([43, 192])
Resized embedding shape:  torch.Size([71, 192])
Vocab expansion verified: 43 -> 71 tokens!


### Step 5: Format Dataset for Training & Validation

In [5]:
from datasets import load_dataset, Audio

TRAIN_MANIFEST = "/kaggle/working/urhobo-tts/data/processed/finetune/train.jsonl"
DEV_MANIFEST   = "/kaggle/working/urhobo-tts/data/processed/finetune/dev.jsonl"

# Load jsonl manifests
dataset = load_dataset("json", data_files={"train": TRAIN_MANIFEST, "eval": DEV_MANIFEST})

# Fix relative audio paths to absolute paths
def fix_audio_path(batch):
    batch["audio"] = [os.path.join("/kaggle/working/urhobo-tts", path) for path in batch["audio"]]
    return batch

dataset = dataset.map(fix_audio_path, batched=True)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(dataset)
print("Sample item:", dataset["train"][0]["id"], dataset["train"][0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/998 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'text', 'duration_sec', 'speaker_id', 'alignment_confidence'],
        num_rows: 998
    })
    eval: Dataset({
        features: ['id', 'audio', 'text', 'duration_sec', 'speaker_id', 'alignment_confidence'],
        num_rows: 101
    })
})
Sample item: GEN_002_001 Kẹnẹ a ma odjúvwu vẹ akpọ́ kugbe obo re vọnrọ ejobi wan.


### Step 6: Launch Fine-Tuning Run
We launch training using the tuned hyperparameters for single-GPU (T4/P100): batch size 16, learning rate 2e-4, checkpoint every 1,000 steps.

In [6]:
import os

OUTPUT_DIR = "/kaggle/working/urhobo_vits_checkpoint"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save extended 70-token Urhobo tokenizer into output directory
tokenizer.save_pretrained(OUTPUT_DIR)

# 2. Launch VITS fine-tuning on Kaggle GPU
!PYTHONPATH="/kaggle/working/finetune-hf-vits" python /kaggle/working/urhobo-tts/scripts/run_urhobo_finetuning.py \
    --model_name_or_path facebook/mms-tts-yor \
    --tokenizer_name /kaggle/working/urhobo-tts/models/urhobo_tokenizer \
    --train_file /kaggle/working/urhobo-tts/data/processed/finetune/train.jsonl \
    --validation_file /kaggle/working/urhobo-tts/data/processed/finetune/dev.jsonl \
    --logging_dir /kaggle/working/urhobo_vits_checkpoint/runs \
    --audio_column_name audio \
    --text_column_name text \
    --output_dir {OUTPUT_DIR} \
    --do_train \
    --do_eval \
    --per_device_train_batch_size 16 \
    --learning_rate 2e-4 \
    --lr_decay 0.999875 \
    --warmup_steps 500 \
    --max_steps 5000 \
    --save_steps 1000 \
    --eval_steps 500 \
    --logging_steps 100 \
    --save_total_limit 3 \
    --fp16 True \
    --override_vocabulary_embeddings True \
    --report_to tensorboard


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
09/17/2026 03:05:45 - WARNING - __main__ - Process rank: -1, device: cuda:0, n_gpu: 2distributed training: False, 16-bits training: True
09/17/2026 03:05:45 - INFO - __main__ - Training/evaluation parameters VITSTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
dd

### Step 7: Synthesize Test Audio & Evaluate Tone Pronunciation

In [7]:
from IPython.display import Audio as PlayAudio, display
from transformers import VitsModel, AutoTokenizer
import torch

OUTPUT_DIR = "/kaggle/working/urhobo_vits_checkpoint"

# 1. Load fine-tuned Urhobo model and extended tokenizer
print(f"Loading fine-tuned Urhobo model from {OUTPUT_DIR}...")
finetuned_model = VitsModel.from_pretrained(OUTPUT_DIR)
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

if torch.cuda.is_available():
    finetuned_model = finetuned_model.to("cuda")
finetuned_model.eval()

# 2. Evaluation sentences covering core vowels, consonants, and tone contrasts
eval_sentences = [
    "Avwanre vwo ẹgba vwọ kẹ Osolobrugwẹ.",
    "Mẹ́vwẹ yen rha cha.",
    "Ọmọ na da rhe vwo ẹghwẹ.",
    "E gbe jẹn orẹmrẹ dia.",
    "Kivie wọ herọ?",
]

print("Synthesizing evaluation phrases with fine-tuned Urhobo weights:")
for sent in eval_sentences:
    inputs = finetuned_tokenizer(sent, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        output = finetuned_model(**inputs).waveform
    audio_arr = output.squeeze().cpu().numpy()
    print(f"\nText: {sent}")
    display(PlayAudio(audio_arr, rate=16000))
    print("-" * 50)


Loading fine-tuned Urhobo model from /kaggle/working/urhobo_vits_checkpoint...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

VitsModel LOAD REPORT from: /kaggle/working/urhobo_vits_checkpoint
Key                                                                             | Status     |  | 
--------------------------------------------------------------------------------+------------+--+-
discriminator.discriminators.{0, 1, 2, 3, 4, 5}.convs.{0, 1, 2, 3, 4, 5}.bias   | UNEXPECTED |  | 
discriminator.discriminators.{0, 1, 2, 3, 4, 5}.convs.{0, 1, 2, 3, 4, 5}.weight | UNEXPECTED |  | 
discriminator.discriminators.{0, 1, 2, 3, 4, 5}.final_conv.weight               | UNEXPECTED |  | 
discriminator.discriminators.{0, 1, 2, 3, 4, 5}.final_conv.bias                 | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Synthesizing evaluation phrases with fine-tuned Urhobo weights:

Text: Avwanre vwo ẹgba vwọ kẹ Osolobrugwẹ.


--------------------------------------------------

Text: Mẹ́vwẹ yen rha cha.


--------------------------------------------------

Text: Ọmọ na da rhe vwo ẹghwẹ.


--------------------------------------------------

Text: E gbe jẹn orẹmrẹ dia.


--------------------------------------------------

Text: Kivie wọ herọ?


--------------------------------------------------


### Step 8: Package & Export Checkpoint Archive

In [8]:
# Compress checkpoint for 1-click download
!zip -r -q /kaggle/working/urhobo_tts_model_checkpoint.zip {OUTPUT_DIR}
print("Checkpoint compressed to /kaggle/working/urhobo_tts_model_checkpoint.zip")

Checkpoint compressed to /kaggle/working/urhobo_tts_model_checkpoint.zip
